In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/credit_default_engineered.csv")
df.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default,AVG_BILL_AMT,AVG_PAY_AMT,CREDIT_UTIL,DELAY_COUNT,MAX_DELAY
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,1,1284.000000,114.833333,0.064200,2,2
1,2,120000,2,2,2,26,-1,2,0,0,...,1000,1000,0,2000,1,2846.166667,833.333333,0.023718,2,2
2,3,90000,2,2,2,34,0,0,0,0,...,1000,1000,1000,5000,0,16942.166667,1836.333333,0.188246,0,0
3,4,50000,2,2,1,37,0,0,0,0,...,1200,1100,1069,1000,0,38555.666667,1398.000000,0.771113,0,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,10000,9000,689,679,0,18223.166667,9841.500000,0.364463,0,0


In [2]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["default", "ID"])
y = df["default"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ("scale", StandardScaler(), X.columns)
    ]
)

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        random_state=42
    )
}

In [5]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

results = []

for name, model in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC_AUC": roc_auc_score(y_test, y_prob)
    })

In [6]:
results_df = pd.DataFrame(results).sort_values("ROC_AUC", ascending=False)
results_df.round(3)

,Model,Accuracy,Precision,Recall,F1,ROC_AUC
2,HistGradientBoosting,0.817,0.655,0.366,0.470,0.778
1,Random Forest,0.812,0.627,0.369,0.464,0.758
0,Logistic Regression,0.808,0.647,0.292,0.402,0.742


## Model Comparison

Three classification models were evaluated.

HistGradientBoosting achieved the strongest overall performance, with the highest ROC-AUC (0.778) and F1 score (0.470).

Random Forest achieved slightly higher recall, while Logistic Regression produced the weakest overall results.

HistGradientBoosting was therefore selected for further evaluation.